# Il gradiente non e' magia: e' una derivata, e si puo' controllare

*Anatomia di un LLM* · capitolo 3, «I mattoni» · nel libro: §3.7 «Il gradiente non è magia» · script `01-05`, `scripts/01_05_autograd.py`

**Dove siamo.** Il gradiente dice in che direzione muovere ogni parametro. PyTorch lo calcola da solo, e qui si controlla che il suo conto coincida con quello fatto a mano: prima su una funzione semplice, poi sulla perdita che il modello userà davvero.

PyTorch calcola da solo di quanto va spostato ogni parametro. Sembra un
oracolo, e invece fa la regola della catena che si studia a scuola. Qui la
rifacciamo a mano tre volte e verifichiamo che i numeri coincidano.

Si esegue con `python3 scripts/01_05_autograd.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/01_05_autograd.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [ ]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [ ]:
import torch
import torch.nn.functional as F

from llm_anatomy.utils import prepara, titolo, verifica

prepara()

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


device(type='cpu')

### Una funzione semplice, derivata a mano

> **Nel libro, §3.7 «Il gradiente non è magia».** Come si sa in che direzione muovere ciascun parametro?

**Cosa guardare.** `x.grad` coincide con la derivata scritta a mano: 38.

In [ ]:
titolo("Una funzione semplice, derivata a mano")
x = torch.tensor(2.0, requires_grad=True)
y = 3 * x ** 3 + 2 * x
y.backward()
analitica = 9 * 2.0 ** 2 + 2
print(f"  y = 3x^3 + 2x   in x = 2   ->  y = {y.item():.1f}")
print(f"  dy/dx analitica = 9x^2 + 2 = {analitica:.1f}")
print(f"  x.grad          = {x.grad.item():.1f}")
verifica(abs(x.grad.item() - analitica) < 1e-6,
         "autograd riproduce la derivata calcolata a mano")


Una funzione semplice, derivata a mano
--------------------------------------
  y = 3x^3 + 2x   in x = 2   ->  y = 28.0
  dy/dx analitica = 9x^2 + 2 = 38.0
  x.grad          = 38.0
  [verifica superata] autograd riproduce la derivata calcolata a mano


### La regola della catena, un pezzo alla volta

*Questa parte non è stampata nel libro.*

**Cosa guardare.** Il prodotto delle due derivate parziali è esattamente `a.grad`: la regola della catena, fatta a mano.

In [ ]:
titolo("La regola della catena, un pezzo alla volta")
a = torch.tensor(1.5, requires_grad=True)
u = torch.sin(a)
v = u ** 2
v.backward()
catena = 2 * torch.sin(torch.tensor(1.5)) * torch.cos(torch.tensor(1.5))
print(f"  v = sin(a)^2   in a = 1,5")
print(f"  dv/du = 2 sin(a) = {2 * torch.sin(torch.tensor(1.5)):.6f}")
print(f"  du/da = cos(a)   = {torch.cos(torch.tensor(1.5)):.6f}")
print(f"  prodotto         = {catena:.6f}")
print(f"  a.grad           = {a.grad.item():.6f}")
verifica(abs(a.grad.item() - catena.item()) < 1e-6,
         "il gradiente e' il prodotto dei due fattori della catena")


La regola della catena, un pezzo alla volta
-------------------------------------------
  v = sin(a)^2   in a = 1,5
  dv/du = 2 sin(a) = 1.994990
  du/da = cos(a)   = 0.070737
  prodotto         = 0.141120
  a.grad           = 0.141120
  [verifica superata] il gradiente e' il prodotto dei due fattori della catena


### Il gradiente della perdita vera

**Cosa guardare.** Le due righe di gradiente sono identiche: probabilità meno uno sul simbolo giusto, diviso per il batch.

In [ ]:
titolo("Il gradiente della perdita vera")
V, B = 5, 3
logits = torch.randn(B, V, requires_grad=True)
bersagli = torch.tensor([0, 3, 4])
perdita = F.cross_entropy(logits, bersagli)
perdita.backward()

prob = logits.softmax(dim=-1)
uno_caldo = F.one_hot(bersagli, V).float()
analitico = (prob - uno_caldo) / B

print(f"  perdita: {perdita.item():.4f}")
print("  gradiente analitico (p - y)/B, prima riga:")
print("   " + " ".join(f"{v:8.5f}" for v in analitico[0].tolist()))
print("  gradiente di autograd, prima riga:")
print("   " + " ".join(f"{v:8.5f}" for v in logits.grad[0].tolist()))
verifica(torch.allclose(logits.grad, analitico, atol=1e-6),
         "il gradiente della cross-entropy e' (probabilita - bersaglio)")
print("  e' la forma che rende l'addestramento stabile: il segnale e' grande")
print("  quando il modello sbaglia con sicurezza, piccolo quando ha ragione")


Il gradiente della perdita vera
-------------------------------
  perdita: 1.9859
  gradiente analitico (p - y)/B, prima riga:
   -0.29806  0.03390  0.08013  0.10742  0.07661
  gradiente di autograd, prima riga:
   -0.29806  0.03390  0.08013  0.10742  0.07661
  [verifica superata] il gradiente della cross-entropy e' (probabilita -
                      bersaglio)
  e' la forma che rende l'addestramento stabile: il segnale e' grande
  quando il modello sbaglia con sicurezza, piccolo quando ha ragione


### Perché bisogna azzerarli ogni volta

**Cosa guardare.** `w.grad` passa da 8 a 16 a 24: i contributi si sommano.

**Prova a cambiare:** subito prima di `perdita_w.backward()` aggiungi `w.grad = None`.

<details><summary>Che cosa succede</summary>

Il gradiente resta 8 a ogni passo, e la verifica fallisce: è proprio l'azzeramento che il ciclo di addestramento fa a ogni giro.

</details>

In [ ]:
titolo("Perche' bisogna azzerarli ogni volta")
w = torch.tensor(1.0, requires_grad=True)
for passo in range(3):
    perdita_w = (w * 2) ** 2
    perdita_w.backward()
    print(f"  passo {passo}: w.grad = {w.grad.item():6.1f}   "
          f"(contributo del singolo passo: 8w = {8 * w.item():.1f})")
verifica(w.grad.item() == 24.0,
         "senza azzeramento i tre contributi si sommano invece di sostituirsi")


Perche' bisogna azzerarli ogni volta
------------------------------------
  passo 0: w.grad =    8.0   (contributo del singolo passo: 8w = 8.0)
  passo 1: w.grad =   16.0   (contributo del singolo passo: 8w = 8.0)
  passo 2: w.grad =   24.0   (contributo del singolo passo: 8w = 8.0)
  [verifica superata] senza azzeramento i tre contributi si sommano invece di
                      sostituirsi


### Staccare un ramo

*Questa parte non è stampata nel libro.*

**Cosa guardare.** Con `detach` il gradiente è 2p invece di 4p: il ramo staccato non conta.

In [ ]:
titolo("Staccare un ramo")
p = torch.tensor(3.0, requires_grad=True)
q = (p * 2).detach() * p
q.backward()
print(f"  q = detach(2p) * p   ->  dq/dp = 2p (non 4p) = {p.grad.item():.1f}")
print("  e' cosi' che si ferma il gradiente dove non deve passare, per")
print("  esempio sui bersagli di una distillazione")


Staccare un ramo
----------------
  q = detach(2p) * p   ->  dq/dp = 2p (non 4p) = 6.0
  e' cosi' che si ferma il gradiente dove non deve passare, per
  esempio sui bersagli di una distillazione
